# 🟡 Medium: Flow Matching Loss (Rectified Flow)

Implement the **Flow Matching / Rectified Flow** training objective.

### Core Idea

Diffusion models learn to reverse a *curved*, hand-designed noising SDE. Flow Matching asks a simpler
question: pick **any** path that connects noise to data, and just regress the **velocity** that moves
a point along it.

Rectified Flow picks the simplest path of all — a **straight line**:

$$x_t = (1-t)\,x_0 + t\,x_1, \qquad t \in [0, 1]$$

with $x_0 \sim \mathcal{N}(0, I)$ (noise) and $x_1$ a real data sample. Differentiate the path:

$$u_t = \frac{d x_t}{d t} = x_1 - x_0$$

The target velocity is **constant in $t$** — the whole schedule/SNR machinery of DDPM disappears. Train
a network $v_\theta$ to regress it:

$$\mathcal{L} = \mathbb{E}_{t \sim U[0,1],\, x_0,\, x_1} \left\| v_\theta(x_t, t) - (x_1 - x_0) \right\|^2$$

At sampling time you just integrate the ODE $\dot{x} = v_\theta(x, t)$ from $t{=}0$ to $t{=}1$. Because
the learned field is nearly straight, a handful of Euler steps is enough — this is why SD3, Flux and
friends moved to it.

**The one bug everyone hits:** `t` has shape `(B,)` while `x` has shape `(B, C, H, W)`. Multiply them
directly and broadcasting silently mixes samples. Reshape `t` to `(B, 1, 1, 1)` first.

### Signature
```python
def flow_matching_loss(model, x0, x1, t):
    # model: callable (x_t, t) -> predicted velocity, same shape as x
    # x0:    noise sample,  shape (B, ...)
    # x1:    data sample,   shape (B, ...)
    # t:     timesteps in [0, 1], shape (B,)
    # returns: scalar MSE loss
    ...
```

### Rules
- Do **NOT** use any built-in flow-matching helper
- `t=0` must give pure noise, `t=1` must give pure data
- The loss must be differentiable w.r.t. the model parameters

### Example
```
x0 = torch.randn(8, 4)        # noise
x1 = torch.randn(8, 4)        # data
t  = torch.rand(8)            # per-sample timestep
loss = flow_matching_loss(lambda x, t: torch.zeros_like(x), x0, x1, t)
# -> equals ((x1 - x0) ** 2).mean(), because the model predicts zero velocity
```

In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def flow_matching_loss(model, x0, x1, t):
    # x0: noise (B, ...)   x1: data (B, ...)   t: (B,) in [0, 1]
    pass  # Replace this

In [ ]:
# 🧪 Test your implementation
x0 = torch.randn(8, 4)
x1 = torch.randn(8, 4)
t = torch.rand(8)

print("zero-velocity model :", flow_matching_loss(lambda x, tt: torch.zeros_like(x), x0, x1, t).item())
print("expected            :", ((x1 - x0) ** 2).mean().item())
print("perfect model       :", flow_matching_loss(lambda x, tt: x1 - x0, x0, x1, t).item(), "(should be ~0)")

In [ ]:
# ✅ SUBMIT — Run this cell to check your solution
from torch_judge import check
check("flow_matching")